In [3]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
import numpy as np

wine = load_wine()
x = wine.data
y = wine.target

scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

x_train, x_test, y_train, y_test = train_test_split(
    x_scaled, y, test_size=0.2, random_state=42
)

base_models = [
    ("dt",  DecisionTreeClassifier(max_depth=4, random_state=42)),
    ("knn", KNeighborsClassifier(n_neighbors=5)),
    ("svc", SVC(probability=True, random_state=42))
]

meta_model = LogisticRegression(max_iter=1000)

stacking = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5
)

stacking.fit(x_train, y_train)

print("stacking classifier - wine dataset")
print("-" * 40)
print("individual model scores:")
for name, model in base_models:
    model.fit(x_train, y_train)
    print(" ", name, "->", round(model.score(x_test, y_test), 3))

print("-" * 40)
print("stacking ->", round(stacking.score(x_test, y_test), 3))

cv = cross_val_score(stacking, x_scaled, y, cv=5)
print("stacking cv ->", round(np.mean(cv), 3))

stacking classifier - wine dataset
----------------------------------------
individual model scores:
  dt -> 0.944
  knn -> 0.944
  svc -> 1.0
----------------------------------------
stacking -> 1.0
stacking cv -> 0.989
